# 🧠 NLP → SQL Expert System
### Converts natural language sentences into SQL queries using spaCy + Experta rule engine

---
**Stack:** Python · spaCy · Experta · Rich · MySQL/PostgreSQL/SQLite/SQL Server  
**Architecture:** NLP Analyser → Fact Declarations → Expert Engine Rules → SQL Output

## 📦 Cell 1 — Imports
Loads all required libraries:
- **`spacy`** — NLP engine for tokenization, POS tagging, and named entity recognition
- **`experta`** — Forward-chaining expert system (rules engine) that fires rules based on declared facts
- **`rich`** — Beautiful terminal UI (colored tables, syntax-highlighted SQL, progress bars)
- **`mysql.connector`** — Optional: live MySQL execution support
- **Standard libs** — `re`, `json`, `datetime`, `collections` for text processing and data handling

In [2]:
import re
import sys
import os
import json
import time
import datetime
import unicodedata
from typing import List, Dict, Optional, Tuple, Any, Set
from collections import defaultdict, OrderedDict
from dataclasses import dataclass, field
from enum import Enum, auto


import collections

if not hasattr(collections, 'Mapping'):
    import collections.abc
    collections.Mapping = collections.abc.Mapping
    collections.MutableMapping = collections.abc.MutableMapping
    collections.Iterable = collections.abc.Iterable
    collections.MutableSet = collections.abc.MutableSet
    collections.Callable = collections.abc.Callable
# spaCy
try:
    import spacy
    from spacy.tokens import Token, Doc, Span
    SPACY_OK = True
except ImportError:
    SPACY_OK = False
    print("  spaCy not installed: pip install spacy && python -m spacy download en_core_web_sm")

# Experta
try:
    from experta import (
        KnowledgeEngine, Rule, Fact, Field,
        MATCH, TEST, AS, NOT, OR, AND, DefFacts
    )
    EXPERTA_OK = True
except ImportError:
    EXPERTA_OK = False
    print(" Experta not installed: pip install experta")

# Rich
try:
    from rich.console import Console
    from rich.table import Table
    from rich.panel import Panel
    from rich.syntax import Syntax
    from rich.tree import Tree
    from rich.text import Text
    from rich.rule import Rule as RichRule
    from rich.columns import Columns
    from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn
    from rich.prompt import Prompt, Confirm
    from rich.align import Align
    from rich import box
    from rich.markdown import Markdown
    RICH_OK = True
    console = Console()
except ImportError:
    RICH_OK = False
    class _FakeConsole:
        def print(self, *a, **k): print(*a)
        def rule(self, *a, **k): print("─" * 60)
        def status(self, *a, **k):
            import contextlib
            return contextlib.nullcontext()
    console = _FakeConsole()

# MySQL (optional)
try:
    import mysql.connector
    MYSQL_OK = True
except ImportError:
    MYSQL_OK = False

## ⚙️ Cell 2 — Enums, Constants & Intent Patterns
Defines the system's core vocabulary and detection rules:
- **`SQLDialect`** — Supported database engines (MySQL, PostgreSQL, SQLite, SQL Server)
- **`SQLIntent`** — The 7 SQL operations the system can generate (CREATE, INSERT, SELECT, UPDATE, DELETE, ALTER, DROP)
- **`INTENT_PATTERNS`** — Regex patterns mapped to each intent; the NLP engine scans input against these to classify what the user wants
- **`APP_TITLE` / `VERSION` / `BANNER`** — App identity metadata and ASCII art displayed in the terminal

In [3]:
class SQLDialect(Enum):
    MYSQL      = "MySQL"
    POSTGRESQL = "PostgreSQL"
    SQLITE     = "SQLite"
    SQLSERVER  = "SQL Server"

class SQLIntent(Enum):
    CREATE     = "CREATE TABLE"
    INSERT     = "INSERT INTO"
    SELECT     = "SELECT"
    UPDATE     = "UPDATE"
    DELETE     = "DELETE"
    ALTER      = "ALTER TABLE"
    DROP       = "DROP TABLE"
    UNKNOWN    = "UNKNOWN"

VERSION   = "3.0.0"
APP_TITLE = "NLP → SQL Expert System"

BANNER = r"""
  ███████╗ ██████╗ ██╗         ███████╗██╗  ██╗██████╗ ███████╗██████╗ ████████╗
  ██╔════╝██╔═══██╗██║         ██╔════╝╚██╗██╔╝██╔══██╗██╔════╝██╔══██╗╚══██╔══╝
  ███████╗██║   ██║██║         █████╗   ╚███╔╝ ██████╔╝█████╗  ██████╔╝   ██║
  ╚════██║██║▄▄ ██║██║         ██╔══╝   ██╔██╗ ██╔═══╝ ██╔══╝  ██╔══██╗   ██║
  ███████║╚██████╔╝███████╗    ███████╗██╔╝ ██╗██║     ███████╗██║  ██║   ██║
  ╚══════╝ ╚══▀▀═╝ ╚══════╝    ╚══════╝╚═╝  ╚═╝╚═╝     ╚══════╝╚═╝  ╚═╝   ╚═╝
"""

# ─── Intent Keywords ───────────────────────────────────────────────────────
INTENT_PATTERNS = {
    SQLIntent.CREATE: [
        r"\b(create|build|make|define|design|generate)\b.*\b(table|schema|database|entity|model)\b",
        r"\b(table|schema)\b.*\b(for|of|about)\b",
        r"\b(store|save|record|keep track of)\b",
        r"\b(columns?|fields?|attributes?)\b",
    ],
    SQLIntent.INSERT: [
        r"\b(insert|add|put|store|save|register|enter|input)\b.*\b(into|data|record|row|value)\b",
        r"\b(new (student|user|employee|record|entry))\b",
        r"\b(add .+? (named?|called?|with name))\b",
    ],
    SQLIntent.SELECT: [
        r"\b(select|get|fetch|find|retrieve|show|list|display|search|look for|query)\b",
        r"\b(who|what|which|where|when|how many)\b.*\b(student|user|employee|record)\b",
        r"\b(all|every)\b.*\b(from|in)\b",
        r"\b(order by|sort|group by|having|join)\b",
    ],
    SQLIntent.UPDATE: [
        r"\b(update|change|modify|edit|set|alter|correct|fix)\b.*\b(record|row|data|value)\b",
        r"\b(update .+? (where|set))\b",
    ],
    SQLIntent.DELETE: [
        r"\b(delete|remove|drop|erase|clear|purge)\b.*\b(record|row|data|entry|from)\b",
        r"\b(delete where)\b",
    ],
    SQLIntent.ALTER: [
        r"\b(add column|drop column|rename|alter|modify column)\b",
        r"\b(change column|add field|remove field)\b",
    ],
    SQLIntent.DROP: [
        r"\b(drop table|delete table|remove table|destroy table)\b",
    ],
}

## 🗂️ Cell 3 — SQL Type Mapping & Entity Profiles
Two critical lookup tables that power intelligent SQL generation:

**`ATTR_TYPE_MAP`** — Maps semantic column name keywords (like `"email"`, `"salary"`, `"status"`) to the correct SQL data type for each dialect. For example:
- `"email"` → `VARCHAR(120) UNIQUE NOT NULL` in MySQL
- `"salary"` → `NUMERIC(12,2)` in PostgreSQL

**`ENTITY_PROFILES`** — Pre-defined column sets for 25+ common entities (student, employee, product, patient, etc.). When you say *"create a table for students"*, the system automatically includes sensible default columns like `student_number`, `gpa`, `enrollment_date` without you needing to list them.

**`ENTITY_STOP`** — A set of common English words that should be ignored during entity extraction (articles, pronouns, prepositions) to avoid false positives like treating "the" or "data" as a table name.

In [4]:
# ─── SQL Type Mapping ──────────────────────────────────────────────────────
ATTR_TYPE_MAP = {
    "id":          {"mysql": "INT UNSIGNED AUTO_INCREMENT PRIMARY KEY",
                    "postgresql": "SERIAL PRIMARY KEY",
                    "sqlite": "INTEGER PRIMARY KEY AUTOINCREMENT",
                    "sqlserver": "INT IDENTITY(1,1) PRIMARY KEY"},
    "name":        {"mysql": "VARCHAR(150) NOT NULL",       "postgresql": "VARCHAR(150) NOT NULL",
                    "sqlite": "TEXT NOT NULL",               "sqlserver": "NVARCHAR(150) NOT NULL"},
    "email":       {"mysql": "VARCHAR(120) UNIQUE NOT NULL","postgresql": "VARCHAR(120) UNIQUE NOT NULL",
                    "sqlite": "TEXT UNIQUE",                 "sqlserver": "NVARCHAR(120) UNIQUE NOT NULL"},
    "phone":       {"mysql": "VARCHAR(25)",                 "postgresql": "VARCHAR(25)",
                    "sqlite": "TEXT",                        "sqlserver": "NVARCHAR(25)"},
    "age":         {"mysql": "TINYINT UNSIGNED",            "postgresql": "SMALLINT",
                    "sqlite": "INTEGER",                     "sqlserver": "TINYINT"},
    "date":        {"mysql": "DATE",                        "postgresql": "DATE",
                    "sqlite": "TEXT",                        "sqlserver": "DATE"},
    "datetime":    {"mysql": "DATETIME DEFAULT CURRENT_TIMESTAMP",
                    "postgresql": "TIMESTAMP DEFAULT NOW()",
                    "sqlite": "TEXT DEFAULT CURRENT_TIMESTAMP",
                    "sqlserver": "DATETIME2 DEFAULT GETDATE()"},
    "salary":      {"mysql": "DECIMAL(12,2)",               "postgresql": "NUMERIC(12,2)",
                    "sqlite": "REAL",                        "sqlserver": "DECIMAL(12,2)"},
    "price":       {"mysql": "DECIMAL(10,2)",               "postgresql": "NUMERIC(10,2)",
                    "sqlite": "REAL",                        "sqlserver": "DECIMAL(10,2)"},
    "amount":      {"mysql": "DECIMAL(14,2)",               "postgresql": "NUMERIC(14,2)",
                    "sqlite": "REAL",                        "sqlserver": "DECIMAL(14,2)"},
    "description": {"mysql": "TEXT",                        "postgresql": "TEXT",
                    "sqlite": "TEXT",                        "sqlserver": "NVARCHAR(MAX)"},
    "address":     {"mysql": "VARCHAR(300)",                "postgresql": "VARCHAR(300)",
                    "sqlite": "TEXT",                        "sqlserver": "NVARCHAR(300)"},
    "status":      {"mysql": "ENUM('active','inactive','pending') DEFAULT 'active'",
                    "postgresql": "VARCHAR(20) DEFAULT 'active'",
                    "sqlite": "TEXT DEFAULT 'active'",
                    "sqlserver": "NVARCHAR(20) DEFAULT 'active'"},
    "score":       {"mysql": "FLOAT",                       "postgresql": "REAL",
                    "sqlite": "REAL",                        "sqlserver": "FLOAT"},
    "url":         {"mysql": "VARCHAR(500)",                "postgresql": "VARCHAR(500)",
                    "sqlite": "TEXT",                        "sqlserver": "NVARCHAR(500)"},
    "image":       {"mysql": "VARCHAR(500)",                "postgresql": "VARCHAR(500)",
                    "sqlite": "TEXT",                        "sqlserver": "NVARCHAR(500)"},
    "password":    {"mysql": "VARCHAR(255) NOT NULL",       "postgresql": "VARCHAR(255) NOT NULL",
                    "sqlite": "TEXT NOT NULL",               "sqlserver": "NVARCHAR(255) NOT NULL"},
    "count":       {"mysql": "INT UNSIGNED DEFAULT 0",      "postgresql": "INTEGER DEFAULT 0",
                    "sqlite": "INTEGER DEFAULT 0",           "sqlserver": "INT DEFAULT 0"},
    "number":      {"mysql": "INT",                         "postgresql": "INTEGER",
                    "sqlite": "INTEGER",                     "sqlserver": "INT"},
    "code":        {"mysql": "VARCHAR(50) UNIQUE NOT NULL", "postgresql": "VARCHAR(50) UNIQUE NOT NULL",
                    "sqlite": "TEXT UNIQUE NOT NULL",        "sqlserver": "NVARCHAR(50) UNIQUE NOT NULL"},
    "title":       {"mysql": "VARCHAR(200) NOT NULL",       "postgresql": "VARCHAR(200) NOT NULL",
                    "sqlite": "TEXT NOT NULL",               "sqlserver": "NVARCHAR(200) NOT NULL"},
    "notes":       {"mysql": "TEXT",                        "postgresql": "TEXT",
                    "sqlite": "TEXT",                        "sqlserver": "NVARCHAR(MAX)"},
    "flag":        {"mysql": "BOOLEAN DEFAULT FALSE",       "postgresql": "BOOLEAN DEFAULT FALSE",
                    "sqlite": "INTEGER DEFAULT 0",           "sqlserver": "BIT DEFAULT 0"},
    "year":        {"mysql": "YEAR",                        "postgresql": "SMALLINT",
                    "sqlite": "INTEGER",                     "sqlserver": "SMALLINT"},
    "default":     {"mysql": "VARCHAR(255)",                "postgresql": "VARCHAR(255)",
                    "sqlite": "TEXT",                        "sqlserver": "NVARCHAR(255)"},
}

# ─── Semantic Entity Profiles ──────────────────────────────────────────────
ENTITY_PROFILES = {
    "student":    ["id","student_number","full_name","email","phone","address","enrollment_date","gpa","status"],
    "professor":  ["id","professor_id","full_name","email","phone","expertise_area","hire_date","office_number"],
    "employee":   ["id","employee_number","full_name","email","phone","department","hire_date","salary","status"],
    "user":       ["id","username","full_name","email","password","phone","created_at","status"],
    "product":    ["id","product_code","title","description","price","stock_count","image","status"],
    "order":      ["id","order_number","order_date","total_amount","status","customer_id"],
    "course":     ["id","course_code","title","description","credits","max_students","status"],
    "subject":    ["id","subject_code","title","description","credits","semester","prerequisites"],
    "department": ["id","dept_code","name","description","manager_id","established_year"],
    "category":   ["id","category_code","name","description","parent_id"],
    "invoice":    ["id","invoice_number","invoice_date","due_date","total_amount","status","customer_id"],
    "customer":   ["id","customer_number","full_name","email","phone","address","created_at"],
    "supplier":   ["id","supplier_code","company_name","contact_name","email","phone","address"],
    "book":       ["id","isbn","title","author_name","publish_year","price","stock_count","category_id"],
    "hospital":   ["id","name","address","phone","director_name","established_year"],
    "patient":    ["id","patient_number","full_name","dob","gender","phone","address","blood_type"],
    "doctor":     ["id","license_number","full_name","specialty","email","phone","hire_date"],
    "room":       ["id","room_number","floor","capacity","room_type","status"],
    "project":    ["id","project_code","title","description","start_date","end_date","budget","status"],
    "task":       ["id","title","description","due_date","priority","status","assigned_to","project_id"],
    "library":    ["id","name","address","phone","opening_hours","manager_id"],
    "lab":        ["id","lab_number","name","location","capacity","equipment_type","supervisor_id"],
    "faculty":    ["id","faculty_code","name","dean_name","established_year","location"],
    "exam":       ["id","exam_code","title","exam_date","duration_minutes","max_score","subject_id"],
    "default":    ["id","name","description","created_at","status"],
}

# ─── Stop Words for entity extraction ──────────────────────────────────────
ENTITY_STOP = {
    "the","a","an","some","any","all","each","every","this","that","these","those",
    "he","she","it","they","we","you","i","me","my","your","his","her","its","their","our",
    "and","or","but","if","then","else","when","where","how","why","what","which","who",
    "be","been","being","have","has","had","do","does","did","will","would","shall","should",
    "may","might","can","could","must","need","dare","used",
    "also","just","very","really","quite","about","above","below","through","into","onto",
    "record","data","information","detail","field","column","attribute","value","row","entry",
}

## 🧩 Cell 4 — Experta Fact Classes
These are the **data containers** of the expert system. Each `Fact` subclass represents one piece of structured knowledge extracted from the user's text. The rule engine pattern-matches against these facts to decide which SQL to generate.

| Fact Class | Purpose |
|---|---|
| `TextInput` | Holds the raw user input + chosen dialect |
| `IntentFact` | The detected SQL operation (CREATE / SELECT / etc.) with confidence score |
| `EntityFact` | A discovered table name with its columns and plural form |
| `ConditionFact` | A WHERE clause condition: column, operator, value |
| `OrderFact` | An ORDER BY directive (column + ASC/DESC) |
| `LimitFact` | A LIMIT N value |
| `JoinFact` | A JOIN between two tables |
| `ColumnSelectFact` | Which columns to SELECT |
| `SetValueFact` | A SET col=val assignment for UPDATE |
| `InsertValueFact` | Column/value pairs for INSERT |
| `SQLResult` | The **output** fact — the final generated SQL string |

In [5]:
class TextInput(Fact):
    """Raw text input from user"""
    text    = Field(str, mandatory=True)
    dialect = Field(str, default="mysql")

class IntentFact(Fact):
    """Detected SQL intent"""
    intent    = Field(str, mandatory=True)   # CREATE, INSERT, SELECT, …
    confidence= Field(float, default=1.0)
    raw_text  = Field(str, default="")

class EntityFact(Fact):
    """A table/entity name discovered in text"""
    name       = Field(str, mandatory=True)
    plural_form= Field(str, default="")
    attributes = Field(tuple, default=lambda: ())
    confidence = Field(float, default=1.0)

class ConditionFact(Fact):
    """A WHERE-clause condition extracted from text"""
    column     = Field(str, mandatory=True)
    operator   = Field(str, default="=")     # = | > | < | LIKE | IN | BETWEEN
    value      = Field(str, default="?")
    connector  = Field(str, default="AND")   # AND | OR

class OrderFact(Fact):
    """ORDER BY extracted"""
    column     = Field(str, mandatory=True)
    direction  = Field(str, default="ASC")

class LimitFact(Fact):
    """LIMIT extracted"""
    n = Field(int, default=10)

class JoinFact(Fact):
    """A JOIN between two tables"""
    table1     = Field(str, mandatory=True)
    table2     = Field(str, mandatory=True)
    join_type  = Field(str, default="INNER JOIN")
    on_col1    = Field(str, default="id")
    on_col2    = Field(str, default="")

class ColumnSelectFact(Fact):
    """Specific columns to SELECT"""
    columns    = Field(tuple, default=lambda: ("*",))

class SetValueFact(Fact):
    """SET col=val for UPDATE"""
    column     = Field(str, mandatory=True)
    value      = Field(str, default="?")

class InsertValueFact(Fact):
    """Values for INSERT"""
    columns    = Field(tuple, default=lambda: ())
    values     = Field(tuple, default=lambda: ())

class SQLResult(Fact):
    """Final generated SQL"""
    sql        = Field(str, mandatory=True)
    intent     = Field(str, mandatory=True)
    table      = Field(str, default="")
    dialect    = Field(str, default="mysql")

## 🔬 Cell 5 — NLP Analyser
The **language understanding layer**. Takes raw English text and extracts structured signals using spaCy and regex patterns.

**Key methods:**
- `detect_intent()` — Scores the text against all `INTENT_PATTERNS` and returns the best-matching `SQLIntent` + a confidence ratio
- `extract_entities()` — Uses spaCy's noun chunking + NER to find table/entity names; falls back to regex if spaCy is unavailable
- `extract_conditions()` — Parses WHERE conditions using patterns like *"where X is Y"*, *"X greater than N"*, *"named X"*, *"with id N"*
- `extract_order()` — Detects ORDER BY directives like *"order by salary desc"* or *"sorted by name"*
- `extract_limit()` — Finds LIMIT values: *"top 10"*, *"first 5 records"*
- `extract_select_columns()` — Identifies explicit column names: *"show me the name and email"*
- `extract_insert_values()` — Parses col=value pairs and quoted strings for INSERT
- `extract_update_set()` — Finds SET assignments: *"set status to inactive"*
- `detect_joins()` — Detects multi-table queries: *"students with their courses"*

In [27]:
from typing import List, Dict, Tuple, Optional
from collections import defaultdict
import re

# ─── ENTITY_STOP الكاملة ───────────────────────────────────────────────────
ENTITY_STOP = {
    "the","a","an","some","any","all","each","every","this","that","these","those",
    "he","she","it","they","we","you","i","me","my","your","his","her","its","their","our",
    "and","or","but","if","then","else","when","where","how","why","what","which","who",
    "be","been","being","have","has","had","do","does","did","will","would","shall","should",
    "may","might","can","could","must","need","dare","used",
    "also","just","very","really","quite","about","above","below","through","into","onto",
    "record","data","information","detail","field","column","attribute","value","row","entry",
    # ✅ مضافات جديدة — منع الأعمدة من تصير entities
    "name","email","phone","gpa","title","price","salary","address",
    "password","status","date","table","number","code","score",
    "description","image","url","flag","year","notes","count",
}

# ─── INTENT_PATTERNS الكاملة ──────────────────────────────────────────────
INTENT_PATTERNS = {
    "CREATE": [
        r"\b(create|build|make|define|design|generate)\b.*\b(table|schema|database|entity|model)\b",
        r"\b(table|schema)\b.*\b(for|of|about)\b",
        r"\b(store|save|record|keep track of)\b",
        r"\b(columns?|fields?|attributes?)\b",
    ],
    "INSERT": [
        r"\b(insert|add|put|store|save|register|enter|input)\b.*\b(into|data|record|row|value)\b",
        r"\b(new (student|user|employee|record|entry))\b",
        r"\b(add .+? (named?|called?|with name))\b",
    ],
    "SELECT": [
        r"\b(select|get|fetch|find|retrieve|show|list|display|search|look for|query)\b",
        r"\b(who|what|which|where|when|how many)\b.*\b(student|user|employee|record)\b",
        r"\b(all|every)\b.*\b(from|in)\b",
        r"\b(order by|sort|group by|having|join)\b",
    ],
    "UPDATE": [
        r"\b(update|change|modify|edit|set|alter|correct|fix)\b.*\b(record|row|data|value)\b",
        r"\b(update .+? (where|set))\b",
    ],
    "DELETE": [
        r"\b(delete|remove|drop|erase|clear|purge)\b.*\b(record|row|data|entry|from)\b",
        r"\b(delete where)\b",
    ],
    "ALTER": [
        r"\b(add column|drop column|rename|alter|modify column)\b",
        r"\b(change column|add field|remove field)\b",
    ],
    "DROP": [
        r"\b(drop table|delete table|remove table|destroy table)\b",
    ],
}


class NLPAnalyser:
    """
    Analyses raw text with spaCy and extracts structured signals:
    intent, entities, conditions, columns, values, etc.
    """

    def __init__(self):
        try:
            import spacy
            self.nlp = spacy.load("en_core_web_sm")
            self.spacy_ok = True
        except Exception:
            self.nlp = None
            self.spacy_ok = False

    # ── Intent Detection ───────────────────────────────────────────────────
    def detect_intent(self, text: str) -> Tuple[str, float]:
        lower = text.lower()

        # ✅ أولوية قصوى لـ CREATE
        if re.search(r'\bcreate\b', lower) or re.search(r'\bbuild\b', lower):
            if re.search(r'\btable\b|\bschema\b|\bdatabase\b', lower):
                return "CREATE", 1.0

        # ✅ أولوية لـ DROP
        if re.search(r'\bdrop\s+table\b|\bdelete\s+table\b|\bremove\s+table\b', lower):
            return "DROP", 1.0

        scores: Dict[str, float] = defaultdict(float)
        for intent, patterns in INTENT_PATTERNS.items():
            for pat in patterns:
                if re.search(pat, lower, re.IGNORECASE):
                    scores[intent] += 1.0

        if not scores:
            return "UNKNOWN", 0.0

        best = max(scores, key=lambda k: scores[k])
        total = sum(scores.values())
        return best, round(scores[best] / total, 2) if total else 0.0

    # ── Entity Extraction ──────────────────────────────────────────────────
    def extract_entities(self, text: str) -> List[Dict]:
        entities = []
        seen = set()

        if self.spacy_ok and self.nlp:
            doc = self.nlp(text)
            for chunk in doc.noun_chunks:
                root = chunk.root
                lemma = root.lemma_.lower()
                if (
                    lemma not in ENTITY_STOP
                    and len(lemma) > 2
                    and root.pos_ in {"NOUN", "PROPN"}
                    and lemma not in seen
                ):
                    seen.add(lemma)
                    plural = root.text if root.tag_ in {"NNS","NNPS"} else lemma + "s"
                    entities.append({
                        "name":      lemma.capitalize(),
                        "lemma":     lemma,
                        "plural":    plural,
                        "is_plural": root.tag_ in {"NNS","NNPS"},
                    })

            for ent in doc.ents:
                if ent.label_ in {"ORG","PERSON","PRODUCT","GPE","FAC","WORK_OF_ART"}:
                    lemma = ent.text.lower().replace(" ", "_")
                    if lemma not in seen:
                        seen.add(lemma)
                        entities.append({
                            "name":      ent.text.replace(" ","_").capitalize(),
                            "lemma":     lemma,
                            "plural":    lemma + "s",
                            "is_plural": False,
                        })
        else:
            # Fallback بدون spaCy
            words = re.findall(r"\b[a-zA-Z]{3,}\b", text)
            for w in words:
                l = w.lower()
                if l not in ENTITY_STOP and l not in seen:
                    seen.add(l)
                    entities.append({
                        "name": l.capitalize(), "lemma": l,
                        "plural": l + "s", "is_plural": False,
                    })

        return entities

    # ── Condition Extraction ───────────────────────────────────────────────
    def extract_conditions(self, text: str) -> List[Dict]:
        conditions = []
        lower = text.lower()

        where_patterns = [
            r"where\s+(\w+)\s+(?:is|=|equals?)\s+['\"]?([^,.\s'\"]+)['\"]?",
            r"(\w+)\s+(?:is|equals?|=)\s+['\"]([^'\"]+)['\"]",
            r"with\s+(\w+)\s+(?:of|=|:)\s+['\"]?([^\s,.]+)['\"]?",
        ]
        for pat in where_patterns:
            for m in re.finditer(pat, lower):
                col, val = m.group(1), m.group(2)
                if col not in ENTITY_STOP:
                    conditions.append({"column": col, "operator": "=", "value": f"'{val}'"})

        comp_patterns = [
            (r"(\w+)\s+(?:greater than|more than|>)\s+(\d+)",      ">"),
            (r"(\w+)\s+(?:less than|fewer than|<)\s+(\d+)",        "<"),
            (r"(\w+)\s+(?:at least|>=|≥)\s+(\d+)",                 ">="),
            (r"(\w+)\s+(?:at most|<=|≤)\s+(\d+)",                  "<="),
            (r"(\w+)\s+between\s+(\d+)\s+and\s+(\d+)",             "BETWEEN"),
            (r"(\w+)\s+(?:contains?|like|includes?)\s+['\"]?(\w+)['\"]?", "LIKE"),
        ]
        for pat, op in comp_patterns:
            for m in re.finditer(pat, lower):
                col = m.group(1)
                if col in ENTITY_STOP:
                    continue
                if op == "BETWEEN":
                    val = f"{m.group(2)} AND {m.group(3)}"
                elif op == "LIKE":
                    val = f"'%{m.group(2)}%'"
                else:
                    val = m.group(2)
                conditions.append({"column": col, "operator": op, "value": val})

        for m in re.finditer(r"(?:named?|called?)\s+['\"]?([A-Za-z]\w+)['\"]?", lower):
            conditions.append({"column": "name", "operator": "=", "value": f"'{m.group(1)}'"})

        for m in re.finditer(r"with\s+id\s+(\d+)", lower):
            conditions.append({"column": "id", "operator": "=", "value": m.group(1)})

        for m in re.finditer(r"(?:status|state)\s+(?:is\s+)?['\"]?(\w+)['\"]?", lower):
            val = m.group(1)
            if val not in ENTITY_STOP:
                conditions.append({"column": "status", "operator": "=", "value": f"'{val}'"})

        return conditions

    # ── ORDER BY Extraction ────────────────────────────────────────────────
    def extract_order(self, text: str) -> List[Dict]:
        orders = []
        lower = text.lower()
        patterns = [
            r"order(?:ed)?\s+by\s+(\w+)\s*(asc|desc)?",
            r"sort(?:ed)?\s+by\s+(\w+)\s*(asc|desc)?",
            r"sorted?\s+(?:in\s+)?(?:ascending|descending)\s+order\s+(?:by|of)\s+(\w+)",
        ]
        for pat in patterns:
            for m in re.finditer(pat, lower):
                col = m.group(1)
                if col in ENTITY_STOP:
                    continue
                try:
                    direction = (m.group(2) or "asc").upper()
                except:
                    direction = "ASC"
                if "descending" in lower or "desc" in lower:
                    direction = "DESC"
                orders.append({"column": col, "direction": direction})
        return orders

    # ── LIMIT Extraction ───────────────────────────────────────────────────
    def extract_limit(self, text: str) -> Optional[int]:
        lower = text.lower()
        for pat in [
            r"(?:limit|top|first|only)\s+(\d+)",
            r"(\d+)\s+(?:records?|rows?|results?|entries)",
        ]:
            m = re.search(pat, lower)
            if m:
                return int(m.group(1))
        return None

    # ── Column SELECT Extraction ───────────────────────────────────────────
    def extract_select_columns(self, text: str) -> List[str]:
        lower = text.lower()
        cols = []
        col_pattern = r"(?:show|display|get|fetch|select|retrieve)\s+(?:me\s+)?(?:the\s+)?(.+?)(?:\s+from|\s+of|\s+where|\s+in|\.|$)"
        m = re.search(col_pattern, lower)
        if m:
            raw = m.group(1)
            parts = re.split(r"[,\s]+and\s+|,\s*|\s+and\s+", raw)
            for p in parts:
                p = p.strip()
                if p and p not in ENTITY_STOP and len(p) > 1:
                    cols.append(p.replace(" ", "_"))
        if not cols or re.search(r"\ball\b|\beverything\b|\*", lower):
            return ["*"]
        return cols if cols else ["*"]

    # ── INSERT Values Extraction ───────────────────────────────────────────
    def extract_insert_values(self, text: str) -> Dict:
        lower = text.lower()
        result = {"columns": [], "values": []}
        pairs = re.findall(
            r"(\w+)\s*(?:is|=|:)\s*['\"]?([^,.\n'\"]+?)['\"]?(?=\s*(?:,|and|\.|$))",
            lower
        )
        for col, val in pairs:
            if col in ENTITY_STOP or col in {"add","insert","into","new","create"}:
                continue
            result["columns"].append(col)
            val = val.strip()
            if re.match(r"^\d+(\.\d+)?$", val):
                result["values"].append(val)
            else:
                result["values"].append(f"'{val}'")

        quoted = re.findall(r"['\"]([^'\"]+)['\"]", text)
        if not result["columns"] and quoted:
            result["columns"] = ["name"]
            result["values"]  = [f"'{quoted[0]}'"]

        return result

    # ── UPDATE SET Extraction ──────────────────────────────────────────────
    def extract_update_set(self, text: str) -> List[Dict]:
        lower = text.lower()
        sets  = []
        for pat in [
            r"(?:set|change|update|modify)\s+(\w+)\s+to\s+['\"]?([^,.\n'\"]+)['\"]?",
            r"(\w+)\s*=\s*['\"]?([^,.\n'\"]+)['\"]?",
        ]:
            for m in re.finditer(pat, lower):
                col, val = m.group(1).strip(), m.group(2).strip()
                if col in ENTITY_STOP:
                    continue
                if re.match(r"^\d+(\.\d+)?$", val):
                    sets.append({"column": col, "value": val})
                else:
                    sets.append({"column": col, "value": f"'{val}'"})
        return sets

    # ── JOIN Detection ─────────────────────────────────────────────────────
    def detect_joins(self, text: str, entities: List[Dict]) -> List[Dict]:
        lower  = text.lower()
        joins  = []
        enames = [e["lemma"] for e in entities]

        join_keywords = {
            "inner join": "INNER JOIN",
            "left join":  "LEFT JOIN",
            "right join": "RIGHT JOIN",
            "join":       "INNER JOIN",
        }
        for kw, jtype in join_keywords.items():
            if kw in lower and len(enames) >= 2:
                joins.append({
                    "table1":    enames[0].capitalize(),
                    "table2":    enames[1].capitalize(),
                    "join_type": jtype,
                    "on_col":    f"{enames[0]}_id",
                })

        if not joins and len(enames) >= 2 and any(
            k in lower for k in ["with their","together with","along with","and their"]
        ):
            joins.append({
                "table1":    enames[0].capitalize(),
                "table2":    enames[1].capitalize(),
                "join_type": "INNER JOIN",
                "on_col":    f"{enames[0]}_id",
            })

        return joins

## ⚡ Cell 6 — Experta Expert Engine (`SQLExpertEngine`)
The **brain of the system** — a forward-chaining rule engine that fires rules based on which facts are present in the knowledge base.

**How it works:**
1. Facts are loaded into the engine's working memory
2. The engine scans all `@Rule` decorators to find rules whose conditions match the current facts
3. Every matching rule fires and produces a `SQLResult` fact containing the final SQL

**Helper methods (internal utilities):**
- `_col_type(col_name)` — Looks up the correct SQL data type for a column by scanning `ATTR_TYPE_MAP`
- `_entity_columns(entity_name)` — Returns the full column list for an entity by merging its `ENTITY_PROFILES` entry with any extra attributes
- `_build_where(conditions)` — Assembles a `WHERE` clause string from a list of `ConditionFact` dicts

**Rules fired per intent:**
- `rule_create_table` — Builds full `CREATE TABLE` with FK constraints + dialect-specific syntax
- `rule_select_simple` / `rule_select_where` / `rule_select_order` / `rule_select_limit` — Four SELECT variants
- `rule_insert` — Builds `INSERT INTO` with explicit columns or generic placeholder
- `rule_update` / `rule_update_where` — UPDATE with optional WHERE condition
- `rule_delete_where` / `rule_delete_all` — DELETE with safety warning if no condition
- `rule_alter_add_column` — ALTER TABLE ADD COLUMN
- `rule_drop_table` — DROP TABLE with ⚠️ warning comment

In [28]:
class SQLExpertEngine(KnowledgeEngine):
    """
    Expert System that generates SQL from declared facts.
    Each rule handles one SQL intent and fires only when conditions are met.
    """

    def __init__(self, dialect: str = "mysql"):
        super().__init__()
        self.dialect    = dialect.lower()
        self.generated  : List[str] = []

    # ── Helper: get SQL type for column ───────────────────────────────────
    def _col_type(self, col_name: str) -> str:
        lower = col_name.lower()
        for key, types in ATTR_TYPE_MAP.items():
            if key in lower:
                return types.get(self.dialect, types["mysql"])
        return ATTR_TYPE_MAP["default"].get(self.dialect, "VARCHAR(255)")

    # ── Helper: get columns for entity ────────────────────────────────────
    def _entity_columns(self, entity_name: str, extra_attrs: tuple = ()) -> List[str]:
        profile_key = entity_name.lower()
        base = ENTITY_PROFILES.get(profile_key, ENTITY_PROFILES["default"])
        merged = list(OrderedDict.fromkeys(base + list(extra_attrs)))
        return merged

    # ── Helper: build WHERE clause ─────────────────────────────────────────
    def _build_where(self, conditions: List) -> str:
        if not conditions:
            return ""
        parts = []
        for i, c in enumerate(conditions):
            conn = "" if i == 0 else f" {c.get('connector','AND')} "
            op   = c.get("operator", "=")
            col  = c.get("column","id")
            val  = c.get("value","?")
            parts.append(f"{conn}{col} {op} {val}")
        return "WHERE " + "".join(parts)

    # ── Rule: CREATE TABLE ─────────────────────────────────────────────────
    @Rule(
        IntentFact(intent="CREATE"),
        EntityFact(name=MATCH.entity_name, attributes=MATCH.attrs)
    )
    def rule_create_table(self, entity_name, attrs):
        cols    = self._entity_columns(entity_name, attrs)
        dialect = self.dialect

        col_defs = []
        fk_lines = []
        for col in cols:
            ctype = self._col_type(col)
            if col.endswith("_id") and col != "id":
                ref_table = col.replace("_id", "").capitalize()
                fk_lines.append(
                    f"  FOREIGN KEY ({col}) REFERENCES {ref_table}(id) ON DELETE SET NULL"
                )
            col_defs.append(f"  {col:<25} {ctype}")

        body = ",\n".join(col_defs)
        if fk_lines:
            body += ",\n" + ",\n".join(fk_lines)

        if dialect in ("mysql", ""):
            engine_clause = "\nENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci"
            sql = (
                f"CREATE TABLE IF NOT EXISTS `{entity_name}` (\n"
                f"{body}\n"
                f"){engine_clause};"
            )
        elif dialect == "postgresql":
            sql = f"CREATE TABLE IF NOT EXISTS {entity_name} (\n{body}\n);"
        elif dialect == "sqlite":
            sql = f"CREATE TABLE IF NOT EXISTS {entity_name} (\n{body}\n);"
        elif dialect == "sqlserver":
            sql = f"IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='{entity_name}')\nCREATE TABLE {entity_name} (\n{body}\n);"
        else:
            sql = f"CREATE TABLE {entity_name} (\n{body}\n);"

        self.declare(SQLResult(sql=sql, intent="CREATE", table=entity_name, dialect=dialect))
        self.generated.append(sql)

    # ── Rule: SELECT * ─────────────────────────────────────────────────────
    @Rule(
        IntentFact(intent="SELECT"),
        EntityFact(name=MATCH.table),
        NOT(ConditionFact()),
        NOT(OrderFact()),
        NOT(LimitFact()),
        ColumnSelectFact(columns=MATCH.cols)
    )
    def rule_select_simple(self, table, cols):
        cols_str = ", ".join(cols) if cols != ("*",) else "*"
        sql = f"SELECT {cols_str}\nFROM {table};"
        self.declare(SQLResult(sql=sql, intent="SELECT", table=table))
        self.generated.append(sql)

    # ── Rule: SELECT with WHERE ────────────────────────────────────────────
    @Rule(
        IntentFact(intent="SELECT"),
        EntityFact(name=MATCH.table),
        AS.cond << ConditionFact(column=MATCH.col, operator=MATCH.op, value=MATCH.val),
        ColumnSelectFact(columns=MATCH.cols)
    )
    def rule_select_where(self, table, cols, col, op, val):
        cols_str = ", ".join(cols) if cols != ("*",) else "*"
        sql = f"SELECT {cols_str}\nFROM {table}\nWHERE {col} {op} {val};"
        self.declare(SQLResult(sql=sql, intent="SELECT", table=table))
        self.generated.append(sql)

    # ── Rule: SELECT with ORDER ────────────────────────────────────────────
    @Rule(
        IntentFact(intent="SELECT"),
        EntityFact(name=MATCH.table),
        OrderFact(column=MATCH.order_col, direction=MATCH.direction),
        ColumnSelectFact(columns=MATCH.cols)
    )
    def rule_select_order(self, table, cols, order_col, direction):
        cols_str = ", ".join(cols) if cols != ("*",) else "*"
        sql = f"SELECT {cols_str}\nFROM {table}\nORDER BY {order_col} {direction};"
        self.declare(SQLResult(sql=sql, intent="SELECT", table=table))
        self.generated.append(sql)

    # ── Rule: SELECT with LIMIT ────────────────────────────────────────────
    @Rule(
        IntentFact(intent="SELECT"),
        EntityFact(name=MATCH.table),
        LimitFact(n=MATCH.limit_n),
        ColumnSelectFact(columns=MATCH.cols)
    )
    def rule_select_limit(self, table, cols, limit_n):
        cols_str = ", ".join(cols) if cols != ("*",) else "*"
        if self.dialect == "sqlserver":
            sql = f"SELECT TOP {limit_n} {cols_str}\nFROM {table};"
        else:
            sql = f"SELECT {cols_str}\nFROM {table}\nLIMIT {limit_n};"
        self.declare(SQLResult(sql=sql, intent="SELECT", table=table))
        self.generated.append(sql)

    # ── Rule: INSERT ───────────────────────────────────────────────────────
    @Rule(
        IntentFact(intent="INSERT"),
        EntityFact(name=MATCH.table),
        InsertValueFact(columns=MATCH.cols, values=MATCH.vals)
    )
    def rule_insert(self, table, cols, vals):
        if cols:
            cols_str = ", ".join(cols)
            vals_str = ", ".join(str(v) for v in vals)
            sql = f"INSERT INTO {table} ({cols_str})\nVALUES ({vals_str});"
        else:
            profile = ENTITY_PROFILES.get(table.lower(), ["name","description"])
            non_id  = [c for c in profile if c != "id"][:5]
            cols_str = ", ".join(non_id)
            vals_str = ", ".join(["?" for _ in non_id])
            sql = f"INSERT INTO {table} ({cols_str})\nVALUES ({vals_str});  -- replace ? with actual values"
        self.declare(SQLResult(sql=sql, intent="INSERT", table=table))
        self.generated.append(sql)

    # ── Rule: UPDATE ───────────────────────────────────────────────────────
    @Rule(
        IntentFact(intent="UPDATE"),
        EntityFact(name=MATCH.table),
        SetValueFact(column=MATCH.set_col, value=MATCH.set_val)
    )
    def rule_update(self, table, set_col, set_val):
        sql = (
            f"UPDATE {table}\n"
            f"SET {set_col} = {set_val}\n"
            f"WHERE id = ?;  -- specify the target row id"
        )
        self.declare(SQLResult(sql=sql, intent="UPDATE", table=table))
        self.generated.append(sql)

    # ── Rule: UPDATE with WHERE ────────────────────────────────────────────
    @Rule(
        IntentFact(intent="UPDATE"),
        EntityFact(name=MATCH.table),
        SetValueFact(column=MATCH.set_col, value=MATCH.set_val),
        ConditionFact(column=MATCH.w_col, operator=MATCH.w_op, value=MATCH.w_val)
    )
    def rule_update_where(self, table, set_col, set_val, w_col, w_op, w_val):
        sql = (
            f"UPDATE {table}\n"
            f"SET {set_col} = {set_val}\n"
            f"WHERE {w_col} {w_op} {w_val};"
        )
        self.declare(SQLResult(sql=sql, intent="UPDATE", table=table))
        self.generated.append(sql)

    # ── Rule: DELETE ───────────────────────────────────────────────────────
    @Rule(
        IntentFact(intent="DELETE"),
        EntityFact(name=MATCH.table),
        ConditionFact(column=MATCH.w_col, operator=MATCH.w_op, value=MATCH.w_val)
    )
    def rule_delete_where(self, table, w_col, w_op, w_val):
        sql = (
            f"DELETE FROM {table}\n"
            f"WHERE {w_col} {w_op} {w_val};"
        )
        self.declare(SQLResult(sql=sql, intent="DELETE", table=table))
        self.generated.append(sql)

    # ── Rule: DELETE all (safe version) ───────────────────────────────────
    @Rule(
        IntentFact(intent="DELETE"),
        EntityFact(name=MATCH.table),
        NOT(ConditionFact())
    )
    def rule_delete_all(self, table):
        sql = (
            f"-- ⚠️  WARNING: This deletes ALL rows!\n"
            f"DELETE FROM {table};  -- Add WHERE clause to be safe"
        )
        self.declare(SQLResult(sql=sql, intent="DELETE", table=table))
        self.generated.append(sql)

    # ── Rule: ALTER TABLE (add column) ────────────────────────────────────
    @Rule(
        IntentFact(intent="ALTER"),
        EntityFact(name=MATCH.table),
        ConditionFact(column=MATCH.col_name, value=MATCH.col_type)
    )
    def rule_alter_add_column(self, table, col_name, col_type):
        sql = f"ALTER TABLE {table}\nADD COLUMN {col_name} {col_type};"
        self.declare(SQLResult(sql=sql, intent="ALTER", table=table))
        self.generated.append(sql)

    # ── Rule: DROP TABLE ───────────────────────────────────────────────────
    @Rule(
        IntentFact(intent="DROP"),
        EntityFact(name=MATCH.table)
    )
    def rule_drop_table(self, table):
        sql = f"-- ⚠️  WARNING: This permanently destroys the table!\nDROP TABLE IF EXISTS {table};"
        self.declare(SQLResult(sql=sql, intent="DROP", table=table))
        self.generated.append(sql)

## 🔄 Cell 7 — SQL Generator (Pipeline Orchestrator)
The **main pipeline controller** that connects all components together. When you call `generator.generate("your text here")`, it runs the full pipeline in sequence:

```
Raw Text
   │
   ▼
NLPAnalyser          ← runs all extract_* methods
   │
   ▼
Fact Declarations    ← IntentFact, EntityFact, ConditionFact, etc.
   │
   ▼
SQLExpertEngine.run() ← fires matching @Rule methods
   │
   ▼
SQLResult Facts      ← collected, deduplicated
   │
   ▼
Result Dict          ← returned with SQL, metadata, timing
```

Also maintains a `history` list of all past queries — used by the export and history features.

In [29]:
class SQLGenerator:
    """
    Orchestrates the full pipeline:
    text → NLP → Facts → Expert Engine → SQL
    """

    def __init__(self, dialect: SQLDialect = SQLDialect.MYSQL):
        self.dialect  = dialect
        self.analyser = NLPAnalyser()
        self.history: List[Dict] = []

    def _dialect_key(self) -> str:
        return self.dialect.value.lower().replace(" ", "")

    def generate(self, text: str) -> Dict[str, Any]:
        """Full pipeline: text → SQL result dict"""
        t0 = time.time()
        d_key = self._dialect_key()

        # ── Step 1: NLP Analysis ──────────────────────────────────────────
        intent, confidence = self.analyser.detect_intent(text)
        entities           = self.analyser.extract_entities(text)
        conditions         = self.analyser.extract_conditions(text)
        order              = self.analyser.extract_order(text)
        limit              = self.analyser.extract_limit(text)
        select_cols        = self.analyser.extract_select_columns(text)
        insert_vals        = self.analyser.extract_insert_values(text)
        update_sets        = self.analyser.extract_update_set(text)
        joins              = self.analyser.detect_joins(text, entities)

        # ── Step 2: Build Expert Engine ───────────────────────────────────
        engine = SQLExpertEngine(dialect=d_key)
        engine.reset()

        # ✅ الإصلاح هون — سطر واحد بدون مسافة زيادة
        intent_name = intent.name if isinstance(intent, SQLIntent) else str(intent)
        engine.declare(IntentFact(intent=intent_name, confidence=confidence, raw_text=text))

        for ent in entities[:3]:
            engine.declare(EntityFact(name=ent["name"], plural_form=ent["plural"], confidence=1.0))

        for cond in conditions[:5]:
            engine.declare(ConditionFact(column=cond["column"], operator=cond["operator"], value=cond["value"]))

        for o in order[:2]:
            engine.declare(OrderFact(column=o["column"], direction=o["direction"]))

        if limit:
            engine.declare(LimitFact(n=limit))

        engine.declare(ColumnSelectFact(columns=tuple(select_cols)))

        if insert_vals["columns"]:
            engine.declare(InsertValueFact(columns=tuple(insert_vals["columns"]), values=tuple(insert_vals["values"])))
        else:
            engine.declare(InsertValueFact())

        for s in update_sets[:3]:
            engine.declare(SetValueFact(column=s["column"], value=s["value"]))

        # ── Step 3: Run Engine ────────────────────────────────────────────
        engine.run()

        # ── Step 4: Collect Results ───────────────────────────────────────
        results = []
        for fact in engine.facts.values():
            if isinstance(fact, SQLResult):
                results.append({"sql": fact["sql"], "intent": fact["intent"], "table": fact["table"]})

        seen_sqls = set()
        unique_results = []
        for r in results:
            if r["sql"] not in seen_sqls:
                seen_sqls.add(r["sql"])
                unique_results.append(r)

        elapsed = round(time.time() - t0, 3)

        result = {
            "input_text":   text,
            "dialect":      self.dialect.value,
            "intent":       intent_name,   # ✅ استخدم intent_name مش intent.name
            "confidence":   confidence,
            "entities":     entities,
            "conditions":   conditions,
            "order":        order,
            "limit":        limit,
            "joins":        joins,
            "results":      unique_results,
            "elapsed_ms":   int(elapsed * 1000),
            "timestamp":    datetime.datetime.now().isoformat(),
        }

        self.history.append(result)
        return result

## 🎨 Cell 8 — Display Engine
Handles all **Rich terminal output** — tables, syntax-highlighted SQL panels, history views, and help trees.

- **`print_banner()`** — Shows the ASCII art logo and app title on startup
- **`print_analysis(result)`** — Renders a Rich table showing the NLP breakdown: intent, confidence, entities, conditions, ORDER BY, LIMIT, JOIN, and processing time
- **`print_sql_results(results, dialect)`** — Renders each generated SQL in a syntax-highlighted panel with per-intent color themes (monokai for CREATE, dracula for SELECT, etc.)
- **`print_history(history)`** — Tabular view of the last 20 queries: timestamp, intent, table name, and truncated input text
- **`print_help()`** — Displays example queries organized by intent as Rich trees

In [30]:
class DisplayEngine:
    """Handles all Rich terminal output"""

    INTENT_COLORS = {
        "CREATE": "bold green",
        "INSERT": "bold blue",
        "SELECT": "bold cyan",
        "UPDATE": "bold yellow",
        "DELETE": "bold red",
        "ALTER":  "bold magenta",
        "DROP":   "bold red",
        "UNKNOWN":"dim white",
    }

    INTENT_ICONS = {
        "CREATE": "🏗️ ", "INSERT": "➕ ", "SELECT": "🔍 ",
        "UPDATE": "✏️ ", "DELETE": "🗑️ ", "ALTER":  "🔧 ",
        "DROP":   "💣 ", "UNKNOWN": "❓ ",
    }

    def print_banner(self):
        console.print(f"[bold cyan]{BANNER}[/bold cyan]")
        console.print(Panel(
            f"[bold white]{APP_TITLE}[/bold white]  [dim]v{VERSION}[/dim]\n"
            "[dim]Type natural language → get SQL instantly[/dim]",
            border_style="cyan", padding=(0, 4)
        ))

    def print_analysis(self, result: Dict):
        intent  = result["intent"]
        color   = self.INTENT_COLORS.get(intent, "white")
        icon    = self.INTENT_ICONS.get(intent, "")

        t = Table(
            title=f"{icon}[{color}]{intent}[/{color}] Analysis",
            box=box.ROUNDED, show_header=True,
            header_style="bold dim", border_style="dim"
        )
        t.add_column("Signal",    style="bold",     min_width=18)
        t.add_column("Detected",  style="cyan",     min_width=40)

        t.add_row("Intent",      f"[{color}]{intent}[/{color}]  (conf: {result['confidence']:.0%})")
        t.add_row("Dialect",     result["dialect"])
        t.add_row("Entities",    ", ".join(e["name"] for e in result["entities"][:5]) or "—")

        if result["conditions"]:
            conds = "; ".join(f"{c['column']} {c['operator']} {c['value']}" for c in result["conditions"][:4])
            t.add_row("Conditions", conds)
        if result["order"]:
            orders = ", ".join(f"{o['column']} {o['direction']}" for o in result["order"])
            t.add_row("ORDER BY", orders)
        if result["limit"]:
            t.add_row("LIMIT", str(result["limit"]))
        if result["joins"]:
            j = result["joins"][0]
            t.add_row("JOIN", f"{j['table1']} ⟶ {j['table2']}  ({j['join_type']})")

        t.add_row("Time", f"{result['elapsed_ms']} ms")
        console.print(t)

    def print_sql_results(self, results: List[Dict], dialect: str):
        if not results:
            console.print(Panel(
                "[yellow]⚠  No SQL generated for this input.\n"
                "Try being more specific (e.g. 'create a table for students')[/yellow]",
                border_style="yellow"
            ))
            return

        console.print()
        console.rule("[bold green]Generated SQL[/bold green]")

        themes = {
            "CREATE": "monokai", "SELECT": "dracula", "INSERT": "github-dark",
            "UPDATE": "zenburn", "DELETE": "gruvbox-dark", "ALTER": "one-dark", "DROP": "gruvbox-dark",
        }

        for i, r in enumerate(results, 1):
            intent = r["intent"]
            color  = self.INTENT_COLORS.get(intent, "white")
            icon   = self.INTENT_ICONS.get(intent, "")
            theme  = themes.get(intent, "monokai")
            console.print(Panel(
                Syntax(r["sql"], "sql", theme=theme, line_numbers=True, word_wrap=True),
                title=f"{icon}[{color}] Statement {i} — {intent} [/{color}]",
                border_style=color.replace("bold ", "") if "bold" in color else color,
                padding=(0, 1)
            ))

    def print_history(self, history: List[Dict]):
        if not history:
            console.print("[dim]No history yet.[/dim]")
            return

        t = Table(title="📜 Query History", box=box.SIMPLE_HEAVY, show_header=True, header_style="bold magenta")
        t.add_column("#",        style="dim",        width=4)
        t.add_column("Time",     style="dim",        width=12)
        t.add_column("Intent",   style="bold",       width=10)
        t.add_column("Table",    style="cyan",       width=18)
        t.add_column("Input",    style="italic dim", min_width=35)

        for i, h in enumerate(history[-20:], 1):
            ts    = h["timestamp"][11:19]
            intent= h["intent"]
            table = h["results"][0]["table"] if h["results"] else "—"
            text  = h["input_text"][:50] + ("…" if len(h["input_text"]) > 50 else "")
            color = self.INTENT_COLORS.get(intent, "white").replace("bold ", "")
            t.add_row(str(i), ts, f"[{color}]{intent}[/{color}]", table, text)
        console.print(t)

    def print_help(self):
        examples = {
            "CREATE": [
                "Create a table for students with name, email and enrollment date",
                "Build a products schema with price, stock and category",
                "Design a doctors table with specialty and license number",
            ],
            "SELECT": [
                "Get all students from the student table",
                "Find students where status is active",
                "Show name and email of employees ordered by salary desc limit 10",
            ],
            "INSERT": [
                "Add a new student named Ahmed with email ahmed@uni.edu",
                "Insert a product called Laptop with price 999",
            ],
            "UPDATE": [
                "Update student set status to inactive where id = 5",
                "Change the salary of employee to 5000 where name is John",
            ],
            "DELETE": [
                "Delete student where id = 10",
                "Remove all inactive users from users table",
            ],
        }
        for intent, queries in examples.items():
            color = self.INTENT_COLORS.get(intent, "white")
            icon  = self.INTENT_ICONS.get(intent, "")
            tree  = Tree(f"{icon}[{color}]{intent}[/{color}]")
            for q in queries:
                tree.add(f"[dim]{q}[/dim]")
            console.print(tree)

## 💾 Cell 9 — Export Engine
Saves the generated SQL and metadata to three different file formats:

- **`export_sql(path)`** — Writes a clean `.sql` file with all generated statements, each preceded by a comment showing the original input text
- **`export_json(path)`** — Saves the full session as structured JSON, including all NLP metadata (intent, entities, conditions, timing) — useful for debugging or downstream processing
- **`export_html(path)`** — Generates a polished dark-themed HTML report with syntax-highlighted SQL cards, intent badges, and dialect/timing metadata — ready to share or view in a browser

All filenames are timestamped (`generated_YYYYMMDD_HHMMSS`) to prevent overwriting previous exports.

In [31]:
class ExportEngine:
    """Exports results to .sql / .json / .html"""

    def __init__(self, history: List[Dict]):
        self.history   = history
        self.timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    def export_sql(self, path: str = ".") -> str:
        fname = os.path.join(path, f"generated_{self.timestamp}.sql")
        lines = [f"-- {APP_TITLE} v{VERSION}\n-- Generated: {self.timestamp}\n\n"]
        for h in self.history:
            lines.append(f"-- Input: {h['input_text'][:80]}\n")
            for r in h["results"]:
                lines.append(r["sql"] + "\n\n")
        with open(fname, "w", encoding="utf-8") as f:
            f.writelines(lines)
        return fname

    def export_json(self, path: str = ".") -> str:
        fname = os.path.join(path, f"generated_{self.timestamp}.json")
        data = {
            "meta": {"tool": APP_TITLE, "version": VERSION, "timestamp": self.timestamp},
            "queries": self.history
        }
        with open(fname, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        return fname

    def export_html(self, path: str = ".") -> str:
        fname = os.path.join(path, f"generated_{self.timestamp}.html")
        color_map = {
            "CREATE":"#22c55e","SELECT":"#06b6d4","INSERT":"#3b82f6",
            "UPDATE":"#eab308","DELETE":"#ef4444","ALTER":"#a855f7","DROP":"#ef4444"
        }
        cards_html = ""
        for h in self.history:
            intent = h["intent"]
            color  = color_map.get(intent, "#94a3b8")
            sqls   = "\n\n".join(r["sql"] for r in h["results"])
            cards_html += f"""
            <div class="card">
              <div class="card-header" style="border-left:4px solid {color}">
                <span class="badge" style="background:{color}">{intent}</span>
                <span class="input-text">{h['input_text'][:100]}</span>
                <span class="meta">{h['dialect']} · {h['elapsed_ms']}ms</span>
              </div>
              <pre class="sql-code">{sqls}</pre>
            </div>"""

        html = f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8"><title>{APP_TITLE}</title>
<style>
  body{{background:#0a0e1a;color:#e2e8f0;font-family:monospace;padding:2rem}}
  .card{{background:#111827;border:1px solid #1f2937;border-radius:10px;margin-bottom:1rem;overflow:hidden}}
  .card-header{{padding:.8rem 1.2rem;display:flex;gap:.8rem;flex-wrap:wrap;background:#0d1117}}
  .badge{{padding:2px 10px;border-radius:999px;font-size:.75rem;font-weight:700;color:#000}}
  .input-text{{flex:1;font-size:.85rem}} .meta{{color:#6b7280;font-size:.75rem;margin-left:auto}}
  .sql-code{{padding:1.2rem;font-size:.82rem;color:#86efac;white-space:pre-wrap}}
</style></head>
<body>
  <h1 style="color:#06b6d4">🧠 {APP_TITLE}</h1>
  <p style="color:#6b7280">v{VERSION} · {self.timestamp} · {len(self.history)} queries</p>
  {cards_html}
</body></html>"""
        with open(fname, "w", encoding="utf-8") as f:
            f.write(html)
        return fname

## 🖥️ Cell 10 — Interactive CLI (REPL)
The **main user interface** — a Read-Eval-Print Loop that accepts commands and natural language input.

**Supported commands:**
| Command | Action |
|---|---|
| `help` | Show example queries organized by intent |
| `history` | Display table of past queries |
| `dialect` | Switch between MySQL / PostgreSQL / SQLite / SQL Server |
| `export` | Save session to `.sql`, `.json`, and `.html` files |
| `clear` | Clear terminal and redisplay banner |
| `exit` / `quit` / `q` | Exit the application |

Any other input is treated as a natural language query and processed through the full pipeline.

In [32]:
class InteractiveCLI:
    """Main REPL loop"""

    DIALECT_MAP = {
        "1": SQLDialect.MYSQL,
        "2": SQLDialect.POSTGRESQL,
        "3": SQLDialect.SQLITE,
        "4": SQLDialect.SQLSERVER,
    }

    def __init__(self):
        self.display   = DisplayEngine()
        self.dialect   = SQLDialect.MYSQL
        self.generator = SQLGenerator(self.dialect)

    def _choose_dialect(self):
        console.print()
        console.rule("[bold]Choose SQL Dialect[/bold]")
        t = Table(box=box.SIMPLE, show_header=False)
        t.add_column("Key",  style="bold cyan", width=4)
        t.add_column("Dialect", style="white")
        for k, d in self.DIALECT_MAP.items():
            t.add_row(f"[{k}]", d.value)
        console.print(t)
        choice = Prompt.ask("  Dialect", choices=["1","2","3","4"], default="1")
        self.dialect   = self.DIALECT_MAP[choice]
        self.generator = SQLGenerator(self.dialect)
        console.print(f"  [green]✓[/green] Using [bold]{self.dialect.value}[/bold]")

    def run(self):
        self.display.print_banner()
        self._choose_dialect()
        console.print()
        console.print(Panel(
            "[dim]Commands:  [bold]help[/bold] · [bold]history[/bold] · "
            "[bold]export[/bold] · [bold]dialect[/bold] · [bold]exit[/bold]\n"
            "Or just type any natural language sentence to generate SQL![/dim]",
            border_style="dim", padding=(0,2)
        ))

        while True:
            console.print()
            try:
                text = Prompt.ask("[bold cyan]You[/bold cyan]").strip()
            except (KeyboardInterrupt, EOFError):
                break

            if not text:
                continue

            lower = text.lower()

            if lower in ("exit", "quit", "q", "bye"):
                console.print("[bold green]Goodbye! 👋[/bold green]")
                break
            elif lower == "help":
                self.display.print_help()
            elif lower == "history":
                self.display.print_history(self.generator.history)
            elif lower == "dialect":
                self._choose_dialect()
            elif lower == "export":
                if not self.generator.history:
                    console.print("[yellow]Nothing to export yet.[/yellow]")
                    continue
                exp = ExportEngine(self.generator.history)
                sql_f  = exp.export_sql()
                json_f = exp.export_json()
                html_f = exp.export_html()
                console.print(f"  [green]✓[/green] SQL  → {sql_f}")
                console.print(f"  [green]✓[/green] JSON → {json_f}")
                console.print(f"  [green]✓[/green] HTML → {html_f}")
            elif lower == "clear":
                console.clear()
                self.display.print_banner()
            else:
                with console.status("[bold green]Analysing text and generating SQL…", spinner="dots2"):
                    result = self.generator.generate(text)
                self.display.print_analysis(result)
                self.display.print_sql_results(result["results"], result["dialect"])

## 🎬 Cell 11 — Demo Mode
A **non-interactive showcase** that runs 10 pre-written example queries automatically, displaying the analysis and generated SQL for each one, then exports the full session.

Run it with:
```bash
python nlp_sql_expert.py --demo
```

Useful for: testing the system quickly, generating a sample HTML report, or demonstrating the tool to others without interactive input.

In [33]:
def run_demo():
    """Run a set of demo queries to showcase the system"""
    display   = DisplayEngine()
    generator = SQLGenerator(SQLDialect.MYSQL)

    display.print_banner()
    console.print(Panel("[bold]🎬  Demo Mode — running example queries[/bold]", border_style="cyan"))

    demo_queries = [
        "Create a table for students with name, email, phone, enrollment date and GPA",
   
    ]

    for query in demo_queries:
        console.print()
        console.rule(f"[dim]{query[:70]}[/dim]")
        with console.status("", spinner="dots"):
            result = generator.generate(query)
        display.print_analysis(result)
        display.print_sql_results(result["results"], result["dialect"])
        time.sleep(0.2)

    console.print()
    console.rule("[bold]Exporting Demo Results[/bold]")
    exp    = ExportEngine(generator.history)
    sql_f  = exp.export_sql()
    json_f = exp.export_json()
    html_f = exp.export_html()
    console.print(f"  [green]✓[/green] SQL  → {sql_f}")
    console.print(f"  [green]✓[/green] JSON → {json_f}")
    console.print(f"  [green]✓[/green] HTML → {html_f}")

    console.print()
    console.print(Panel(
        f"[bold green]✅  Demo complete![/bold green]\n"
        f"[dim]{len(demo_queries)} queries processed · {len(generator.history)} results saved[/dim]",
        border_style="green", padding=(1,4)
    ))

## 🚀 Cell 12 — Entry Point
The **program entry point**. When the script is run directly (not imported as a module):
- If `--demo` flag is passed → runs `run_demo()` (non-interactive showcase)
- Otherwise → launches `InteractiveCLI().run()` (the full REPL)

This is the standard Python `if __name__ == "__main__"` guard pattern.

In [ ]:
if __name__ == "__main__":
    if "--demo" in sys.argv:
        run_demo()
    else:
        cli = InteractiveCLI()
        cli.run()

  ███████╗ ██████╗ ██╗         ███████╗██╗  ██╗██████╗ ███████╗██████╗ ████████╗
  ██╔════╝██╔═══██╗██║         ██╔════╝╚██╗██╔╝██╔══██╗██╔════╝██╔══██╗╚══██╔══╝
  ███████╗██║   ██║██║         █████╗   ╚███╔╝ ██████╔╝█████╗  ██████╔╝   ██║
  ╚════██║██║▄▄ ██║██║         ██╔══╝   ██╔██╗ ██╔═══╝ ██╔══╝  ██╔══██╗   ██║
  ███████║╚██████╔╝███████╗    ███████╗██╔╝ ██╗██║     ███████╗██║  ██║   ██║
  ╚══════╝ ╚══▀▀═╝ ╚══════╝    ╚══════╝╚═╝  ╚═╝╚═╝     ╚══════╝╚═╝  ╚═╝   ╚═╝

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│    NLP → SQL Expert System  v3.0.0                                                                              │
│    Type natural language → get SQL instantly                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────── Choose SQL Dialect ────────────────────────────────────────────────

 [1]    MySQL       
  [2]    PostgreSQL  
  [3]    SQLite      
  [4]    SQL Server 

Dialect [1/2/3/4] (1):

...

                        🏗️ CREATE Analysis                        
╭────────────────────┬──────────────────────────────────────────╮
│ Signal             │ Detected                                 │
├────────────────────┼──────────────────────────────────────────┤
│ Intent             │ CREATE  (conf: 100%)                     │
│ Dialect            │ MySQL                                    │
│ Entities           │ Student, Gpa                             │
│ Time               │ 32 ms                                    │
╰────────────────────┴──────────────────────────────────────────╯

────────────────────────────────────────────────── Generated SQL ──────────────────────────────────────────────────

╭─────────────────────────────────────────── 🏗️  Statement 1 — CREATE  ────────────────────────────────────────────╮
│   1 CREATE TABLE IF NOT EXISTS `Gpa` (                                                                          │
│   2   id                        INT UNSIGNED AUTO_INCREMENT PRIMARY KEY,                                        │
│   3   name                      VARCHAR(150) NOT NULL,                                                          │
│   4   description               TEXT,                                                                           │
│   5   created_at                VARCHAR(255),                                                                   │
│   6   status                    ENUM('active','inactive','pending') DEFAULT 'active'                            │
│   7 )                                                                                                           │
│   8 ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🏗️  Statement 2 — CREATE  ────────────────────────────────────────────╮
│    1 CREATE TABLE IF NOT EXISTS `Student` (                                                                     │
│    2   id                        INT UNSIGNED AUTO_INCREMENT PRIMARY KEY,                                       │
│    3   student_number            INT,                                                                           │
│    4   full_name                 VARCHAR(150) NOT NULL,                                                         │
│    5   email                     VARCHAR(120) UNIQUE NOT NULL,                                                  │
│    6   phone                     VARCHAR(25),                                                                   │
│    7   address                   VARCHAR(300),                                                                  │
│    8   enrollment_date           DATE,                                                                          │
│    9   gpa                       VARCHAR(255),                                                                  │
│   10   status                    ENUM('active','inactive','pending') DEFAULT 'active'                           │
│   11 )                                                                                                          │
│   12 ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

You: